# Random survival forest outputs and leaf relation

Survival risk, survival, and hazard outputs stay native; `survival_leaf` is separately named and evaluation must use censoring-safe metrics.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from forestgap import Proximity
from forestgap.adapters import (
    censoring_safe_survival_metrics, native_survival_outputs,
)

rng = np.random.default_rng(RANDOM_STATE)
X = rng.normal(size=(N_SAMPLES, 5))
event_time = rng.exponential(np.exp(-.7 * X[:, 0]) * 8.0)
censor_time = rng.exponential(11.0, size=N_SAMPLES)
event = event_time <= censor_time
observed_time = np.minimum(event_time, censor_time) + .05
y = np.array(list(zip(event, observed_time)), dtype=[("event", "?"), ("time", "<f8")])
train_ids, test_ids = train_test_split(
    np.arange(N_SAMPLES), test_size=.3, stratify=event,
    random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = X[train_ids], X[test_ids], y[train_ids], y[test_ids]
try:
    from sksurv.ensemble import RandomSurvivalForest
except ImportError:
    RandomSurvivalForest = None

if RandomSurvivalForest is None:
    result = {"status": "skipped", "reason": "install forestgap[survival]"}
else:
    forest = RandomSurvivalForest(
        n_estimators=N_ESTIMATORS, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=1,
    ).fit(X_train, y_train)
    proximity = Proximity(forest, relation="survival_leaf").attach(X_train, y_train)
    outputs = native_survival_outputs(forest, X_test[:3])
    metrics = censoring_safe_survival_metrics(forest, X_test, y_train, y_test)
    result = {
        "status": "ok",
        "train_events": int(np.count_nonzero(y_train["event"])),
        "test_censored": int(np.count_nonzero(~y_test["event"])),
        "risk_shape": outputs["risk"].shape,
        "survival_functions": len(outputs["survival"]),
        "hazard_functions": len(outputs["cumulative_hazard"]),
        "relation_shape": proximity.prox_extend(X_test).shape,
        "ipcw_concordance": metrics["ipcw_concordance"],
        "integrated_brier_score": metrics["integrated_brier_score"],
        "excluded_outside_train_support": metrics["n_excluded_outside_train_support"],
    }
result

{'status': 'ok',
 'train_events': 29,
 'test_censored': 12,
 'risk_shape': (3,),
 'survival_functions': 3,
 'hazard_functions': 3,
 'relation_shape': (24, 56),
 'ipcw_concordance': 0.69859747666098,
 'integrated_brier_score': 0.20777954812177826,
 'excluded_outside_train_support': 0}